# skills package updater

Tests `skills_lib.py` and regenerates `package.yml` from the live
opencode skills directory. Run all cells top-to-bottom after adding or
removing skills.

In [ ]:
import importlib
from pathlib import Path

import skills_lib
importlib.reload(skills_lib)

skills = skills_lib.collect_skills()
print(f"Found {len(skills)} skills under {skills_lib.DEFAULT_SKILLS_DIR}")
skills[:3]

In [ ]:
# Sanity checks: every skill has a name and description
assert len(skills) > 0, "no skills found"
missing = [s["id"] for s in skills if not s["name"] or not s["description"]]
assert not missing, f"skills missing name/description: {missing}"
print("All skills have name + description ✓")

In [ ]:
# Preview the :skills listing (exactly what espanso will expand)
print(skills_lib.format_skill_list(skills))

In [ ]:
# Test lookups: exact id, bare name, nested id, and a miss
for q in ["brainstorming", "tdd", "engineering/tdd", "does-not-exist"]:
    print(f"--- get '{q}' ---")
    print(skills_lib.lookup_skill(q, skills))
    print()

In [ ]:
# Regenerate package.yml
out = skills_lib.generate_package_yml(skills)
print(f"Wrote {out}")
print(out.read_text())

In [ ]:
# End-to-end: run the same CLI commands espanso will invoke
import subprocess, sys

for args in [["list"], ["get", "brainstorming"]]:
    r = subprocess.run(
        [sys.executable, "skills_lib.py", *args],
        capture_output=True, text=True, cwd=Path.cwd(),
    )
    assert r.returncode == 0, r.stderr
    print(f"$ skills_lib.py {' '.join(args)}")
    print(r.stdout)
print("CLI e2e ✓")